In [ ]:
import torch
import torch.nn as nn
import sys
import os

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
# Install ultralytics (uncomment if needed)
# !pip install ultralytics -q

In [ ]:
# Define Scale-Adaptive CBAM modules
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d(1)
        self.pool_m = nn.AdaptiveMaxPool2d(1)
        r = max(channels // reduction, 1)
        self.fc = nn.Sequential(nn.Conv2d(channels, r, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(r, channels, 1, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.fc(self.pool_h(x)) + self.fc(self.pool_m(x)))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return x * self.sigmoid(self.conv(torch.cat([torch.mean(x,1,True), torch.max(x,1,True)[0]], 1)))

class ScaleAdaptiveCBAM(nn.Module):
    """Scale-Adaptive CBAM: dynamic reduction ratio based on channel depth.
    - <=64ch: reduction=4, <=128ch: reduction=8
    - <=256ch: reduction=12, >256ch: reduction=16
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        self.kernel_size = kernel_size
        self.ca = None
        self.sa = None
    def _get_adaptive_reduction(self, channels):
        if channels <= 64: return 4
        elif channels <= 128: return 8
        elif channels <= 256: return 12
        else: return 16
    def forward(self, x):
        if self.ca is None:
            c = x.shape[1]
            reduction = self._get_adaptive_reduction(c)
            self.ca = ChannelAttention(c, reduction).to(x.device)
            self.sa = SpatialAttention(self.kernel_size).to(x.device)
        x = self.ca(x)
        x = self.sa(x)
        return x

In [ ]:
# Register Scale-Adaptive CBAM with ultralytics
import ultralytics.nn.tasks as tasks
import ultralytics.nn.modules as modules

for cls in [ScaleAdaptiveCBAM, ChannelAttention, SpatialAttention]:
    setattr(tasks, cls.__name__, cls)
    setattr(modules, cls.__name__, cls)
    for s in ("block","conv","head"):
        if hasattr(modules,s):
            setattr(getattr(modules,s), cls.__name__, cls)
    sys.modules['__main__'].__dict__[cls.__name__] = cls

torch.serialization.add_safe_globals([ScaleAdaptiveCBAM, ChannelAttention, SpatialAttention])
print("ScaleAdaptiveCBAM registered!")

In [ ]:
# UPDATE THIS PATH to your local dataset location
DATASET = "path/to/your/helmet-detection-dataset/archive7_extracted"

# Create data.yaml
content = f"""train: {DATASET}/images/train
val: {DATASET}/images/val
test: {DATASET}/images/test
nc: 2
names: ['helmet', 'non-helmet']
"""
with open("data.yaml", "w") as f:
    f.write(content)

# Scale-Adaptive CBAM YAML architecture
yaml_text = """nc: 2
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 1, ScaleAdaptiveCBAM, [7]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [19, 1, ScaleAdaptiveCBAM, [7]]
  - [22, 1, ScaleAdaptiveCBAM, [7]]
  - [[16, 23, 24], 1, Detect, [nc]]
"""
with open("scale_adaptive_cbam.yaml", "w") as f:
    f.write(yaml_text)

print("Both YAML files created!")

In [ ]:
from ultralytics import YOLO

RUN_NAME = "scale_adaptive_cbam"
LAST_PT = f"runs/{RUN_NAME}/weights/last.pt"

if os.path.exists(LAST_PT):
    print(f"Resuming from checkpoint: {LAST_PT}")
    model = YOLO(LAST_PT)
    model.train(resume=True, device=0)
else:
    print("Starting fresh training...")
    model = YOLO("scale_adaptive_cbam.yaml")
    model.load("yolov8n.pt")
    model.train(
        data="data.yaml",
        epochs=100,
        batch=16,
        imgsz=640,
        optimizer="SGD",
        lr0=0.01,
        cos_lr=True,
        amp=True,
        close_mosaic=10,
        patience=50,
        workers=2,
        save=True,
        save_period=10,
        project="runs",
        name=RUN_NAME,
        exist_ok=True,
        plots=True,
        device=0  # RTX 4060
    )

print("TRAINING DONE!")